# Import required libraries

In [20]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, make_scorer, fbeta_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

# Prepare for 

In [14]:
# Define features and target variable
feature = [
    "Stop:Station code", 
    "Hour",
    "DayOfWeek_sin",
    "DayOfWeek_cos", 
    "Month_sin", 
    "Month_cos", 
    "IsWeekend", 
    "RushHour",
    "StationTraffic",
    "Temperature",
    "Humidity",
    "Air Pressure",
    "Horizontal Visibility",
    "Cloud Cover",
    "Precipitation Amount",
    "Precipitation Duration",
    "Hourly Mean Wind Speed",
    "Max Wind Speed",
    "Fog",
    "Rainfall",
    "Snowfall",
    "Thunder",
    "Hail",
    "Service:Type"
    ]

target = "is_cancelled"

In [15]:
# Define file paths for each dataset split
files = {
    "Train": "train.csv",
    "Validation": "validation.csv",
    "Test": "test.csv"
}

# Read and process each dataset split in chunks
for split_name, file_path in files.items():
    print(f"\n{split_name} chunks:")
    total_rows = 0

    for i, chunk in enumerate(
        pd.read_csv(file_path, chunksize=1000000),
        start=1
    ):
        X_chunk = chunk[feature]
        y_chunk = chunk[target]
        total_rows += len(chunk)

        print(
            f"{split_name} chunk {i}: {chunk.shape}"
        )

    print(f"{split_name} total rows read: {total_rows:,}")


Train chunks:
Train chunk 1: (1000000, 26)
Train chunk 2: (1000000, 26)
Train chunk 3: (1000000, 26)
Train chunk 4: (1000000, 26)
Train chunk 5: (1000000, 26)
Train chunk 6: (1000000, 26)
Train chunk 7: (1000000, 26)
Train chunk 8: (1000000, 26)
Train chunk 9: (1000000, 26)
Train chunk 10: (1000000, 26)
Train chunk 11: (1000000, 26)
Train chunk 12: (1000000, 26)
Train chunk 13: (1000000, 26)
Train chunk 14: (1000000, 26)
Train chunk 15: (1000000, 26)
Train chunk 16: (1000000, 26)
Train chunk 17: (1000000, 26)
Train chunk 18: (1000000, 26)
Train chunk 19: (1000000, 26)
Train chunk 20: (1000000, 26)
Train chunk 21: (1000000, 26)
Train chunk 22: (1000000, 26)
Train chunk 23: (1000000, 26)
Train chunk 24: (1000000, 26)
Train chunk 25: (1000000, 26)
Train chunk 26: (1000000, 26)
Train chunk 27: (1000000, 26)
Train chunk 28: (1000000, 26)
Train chunk 29: (1000000, 26)
Train chunk 30: (1000000, 26)
Train chunk 31: (1000000, 26)
Train chunk 32: (1000000, 26)
Train chunk 33: (1000000, 26)
Trai

In [ ]:
usecols = feature + [target]

train_df = pd.read_csv("train.csv", usecols=usecols)
validation_df = pd.read_csv("validation.csv", usecols=usecols)
test_df = pd.read_csv("test.csv", usecols=usecols)

for df in (train_df, validation_df, test_df):
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns
    df[float_cols] = df[float_cols].astype("float32")
    df[int_cols] = df[int_cols].astype("int32")

train_df = pd.get_dummies(
    train_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)
validation_df = pd.get_dummies(
    validation_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)
test_df = pd.get_dummies(
    test_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)

validation_df = validation_df.reindex(columns=train_df.columns, fill_value=0)
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

X_train, y_train = train_df.drop(columns=[target]).astype("float32"), train_df[target].astype("int8")
X_val, y_val = validation_df.drop(columns=[target]).astype("float32"), validation_df[target].astype("int8")
X_test, y_test = test_df.drop(columns=[target]).astype("float32"), test_df[target].astype("int8")

# Baseline: Logistic Regression

In [29]:
# Baseline: Logistic Regression from existing prepared data (no re-reading CSV).
column_means = X_train.mean().astype("float32")
model = SGDClassifier(loss="log_loss", random_state=42)

# Train in chunks to avoid memory issues
for i, start in enumerate(range(0, len(X_train), 1_000_000), start=1):
    X_chunk = X_train.iloc[start:start + 1_000_000]
    y_chunk = y_train.iloc[start:start + 1_000_000]

    X_chunk_np = X_chunk.fillna(column_means).to_numpy(dtype=np.float32, copy=False)
    y_chunk_np = y_chunk.to_numpy(copy=False)

    if i == 1:
        model.partial_fit(X_chunk_np, y_chunk_np, classes=np.array([0, 1]))
    else:
        model.partial_fit(X_chunk_np, y_chunk_np)

    print(f"Train chunk {i}: {X_chunk.shape}")

# Evaluate on validation and test sets in chunks to avoid memory issues
def evaluate_split(name, X, y):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for start in range(0, len(X), 1_000_000):
        X_chunk = X.iloc[start:start + 1_000_000]
        y_chunk = y.iloc[start:start + 1_000_000]

        X_chunk_np = X_chunk.fillna(column_means).to_numpy(dtype=np.float32, copy=False)
        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(model.predict(X_chunk_np))
        y_prob_parts.append(model.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    print(f"\n{name}")
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("ROC-AUC:", round(roc_auc_score(y_true_all, y_prob_all), 4))

evaluate_split("Validation", X_val, y_val)
evaluate_split("Test", X_test, y_test)

Train chunk 1: (1000000, 316)
Train chunk 2: (1000000, 316)
Train chunk 3: (1000000, 316)
Train chunk 4: (1000000, 316)
Train chunk 5: (1000000, 316)
Train chunk 6: (1000000, 316)
Train chunk 7: (1000000, 316)
Train chunk 8: (1000000, 316)
Train chunk 9: (1000000, 316)
Train chunk 10: (1000000, 316)
Train chunk 11: (1000000, 316)
Train chunk 12: (1000000, 316)
Train chunk 13: (1000000, 316)
Train chunk 14: (1000000, 316)
Train chunk 15: (1000000, 316)
Train chunk 16: (1000000, 316)
Train chunk 17: (1000000, 316)
Train chunk 18: (1000000, 316)
Train chunk 19: (1000000, 316)
Train chunk 20: (1000000, 316)
Train chunk 21: (1000000, 316)
Train chunk 22: (1000000, 316)
Train chunk 23: (1000000, 316)
Train chunk 24: (1000000, 316)
Train chunk 25: (1000000, 316)
Train chunk 26: (1000000, 316)
Train chunk 27: (1000000, 316)
Train chunk 28: (1000000, 316)
Train chunk 29: (1000000, 316)
Train chunk 30: (1000000, 316)
Train chunk 31: (1000000, 316)
Train chunk 32: (1000000, 316)
Train chunk 33: (

# Random Forest
Due to the computational size of the full dataset, a stratified sampling procedure was applied prior to model training. Sampling was performed separately for each year and cancellation class to preserve both the temporal distribution and class imbalance structure of the original dataset. This approach reduced computational requirements while maintaining representative operational patterns across the analysis period.

In [40]:
# Random Forest from existing prepared data (no re-reading CSV).
# Train on small random sample to avoid memory issues
sample_size = 250_000
indices = np.arange(len(X_train), dtype=np.int32)
np.random.seed(42)
np.random.shuffle(indices)
indices = indices[:sample_size]

X_train_sample = X_train.iloc[indices].astype("float32").to_numpy(copy=False)
y_train_sample = y_train.iloc[indices].to_numpy(copy=False)

del indices

rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=6,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_sample, y_train_sample)
print(f"Random Forest trained on {len(y_train_sample):,} samples")

# Evaluate on validation and test sets in chunks to avoid memory issues
def evaluate_split_rf(name, X, y):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for start in range(0, len(X), 1_000_000):
        X_chunk = X.iloc[start:start + 1_000_000]
        y_chunk = y.iloc[start:start + 1_000_000]

        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)
        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(rf.predict(X_chunk_np))
        y_prob_parts.append(rf.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    print(f"\n{name}")
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("ROC-AUC:", round(roc_auc_score(y_true_all, y_prob_all), 4))

evaluate_split_rf("Validation", X_val, y_val)
evaluate_split_rf("Test", X_test, y_test)

# Feature importances
feature_cols = X_train.columns
importances = pd.Series(
    rf.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\nTop 20 Feature Importances:")
print(importances.head(20))

MemoryError: Unable to allocate 191. MiB for an array with shape (50000000,) and data type int32

# XGBoost: Extreme Gradient Boosting 

In [ ]:
# scale_pos_weight = (
#     y_sample.value_counts()[0] /
#     y_sample.value_counts()[1]
# )

# xgb = XGBClassifier(
#     n_estimators=100,
#     max_depth=6,
#     learning_rate=0.1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     scale_pos_weight=scale_pos_weight,
#     random_state=42,
#     n_jobs=-1,
#     eval_metric="logloss",
#     tree_method="hist"
# )

# f2_scorer = make_scorer(fbeta_score, beta=2)

# scores = cross_val_score(
#     xgb,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=f2_scorer,
#     n_jobs=1
# )

# print("F2 scores:", scores)
# print("Mean F2:", scores.mean())

# xgb.fit(X_sample, y_sample)

# importances = pd.Series(
#     xgb.feature_importances_,
#     index=X_sample.columns
# ).sort_values(ascending=False)

# print(importances)

In [ ]:
# from sklearn.model_selection import cross_validate
# from sklearn.metrics import make_scorer, precision_score, recall_score, fbeta_score

# # Scorers
# precision = make_scorer(precision_score)
# recall = make_scorer(recall_score)
# f2 = make_scorer(fbeta_score, beta=2)

# scoring = {
#     "precision": precision,
#     "recall": recall,
#     "f2": f2
# }

# # Random Forest

# rf_scores = cross_validate(
#     rf,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=scoring,
#     n_jobs=1
# )

# print("=== Random Forest ===")
# print("Mean Precision:", rf_scores["test_precision"].mean())
# print("Mean Recall:", rf_scores["test_recall"].mean())
# print("Mean F2:", rf_scores["test_f2"].mean())

# # XGBOOST

# xgb_scores = cross_validate(
#     xgb,
#     X_sample,
#     y_sample,
#     cv=sgkfold,
#     groups=groups_sample,
#     scoring=scoring,
#     n_jobs=1
# )

# print("\n=== XGBoost ===")
# print("Mean Precision:", xgb_scores["test_precision"].mean())
# print("Mean Recall:", xgb_scores["test_recall"].mean())
# print("Mean F2:", xgb_scores["test_f2"].mean())